# Устанавливаем библиотеки

In [ ]:
# !pip install peft
# !pip install trl
# !pip install accelerate
# !pip install bitsandbytes
# !pip install zstandard
# !pip install torchao --upgrade

# Проверяем доступность GPU и ее характеристики

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Память GPU, GiB:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
    )

PyTorch: 2.11.0+cpu
CUDA доступна: False


# Минимальная конфигурация PEFT

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(base_model, config)

model.print_trainable_parameters()

# Загружаем датасет для дообучения моделей

In [2]:
import io
import json
import zstandard

from huggingface_hub import hf_hub_download
from datasets import Dataset

# 1. Скачиваем архив напрямую из Hugging Face Hub
file_path = hf_hub_download(repo_id="IlyaGusev/ru_turbo_alpaca", filename="ru_turbo_alpaca.jsonl.zst", repo_type="dataset")

# 2. Распаковываем Zstandard и читаем JSONL
rows = []

with open(file_path, "rb") as compressed_file:
    decompressor = zstandard.ZstdDecompressor()

    with decompressor.stream_reader(compressed_file) as reader:
        with io.TextIOWrapper(reader, encoding="utf-8") as text_reader:
            for line in text_reader:
                if line.strip():
                    rows.append(json.loads(line))

# 3. Создаем Dataset без скрипта загрузки
dataset = Dataset.from_list(rows)

print(dataset)
print(dataset[0])

ru_turbo_alpaca.jsonl.zst: reconstructing file:   0%|          |  0.00B / 14.6MB            

ru_turbo_alpaca.jsonl.zst: downloading bytes:           |  0.00B            

Dataset({
    features: ['instruction', 'input', 'output', 'label', 'all_labels', 'overlap', 'agreement', 'alternative_output'],
    num_rows: 29822
})
{'instruction': 'Опишите, как сделать горшок из глины.', 'input': '', 'output': 'Для изготовления горшка из глины сначала необходимо взять глину и размешать ее водой до состояния пластилина. Затем нужно сформировать глиняный комок нужной формы и тщательно откачать влагу из глины, чтобы она стала твердой. Горшок нужно обжечь в духовке при высокой температуре, чтобы он стал прочным и непроницаемым.', 'label': 'bad_output', 'all_labels': ['bad_output', 'bad_output'], 'overlap': 2, 'agreement': 1.0, 'alternative_output': 'Для изготовления горшка из глины нужно выполнить следующие шаги:\n\n1. Подготовить глину: глину необходимо хорошо промыть и измельчить. Если глина слишком сухая, ее нужно замочить.\n\n2. Сформировать горшок: возьмите кусок глины и разминайте его в руках, сделав шарик. Потом начинайте вращать шарик между ладонями, чтоб сфор

In [3]:
def format_example(example):
    instruction = example["instruction"]
    input_text = example.get("input", "") or ""
    answer = example["output"]

    if input_text.strip():
        prompt = (
            f"Инструкция: {instruction}\n"
            f"Входные данные: {input_text}\n"
            "Ответ:"
        )
    else:
        prompt = f"Инструкция: {instruction}\nОтвет:"

    return {
        "text": prompt + " " + answer
    }


dataset = dataset.map(format_example)
dataset = dataset.select_columns(["text"])
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset[0]["text"])

Map:   0%|          | 0/29822 [00:00<?, ? examples/s]

Инструкция: Предложи несколько способов решения проблемы отсутствия мотивации.
Ответ: 1. Поставьте перед собой конкретные цели и награды за их достижение. 2. Изучайте новые темы, которые могут вас заинтересовать. 3. Попробуйте изменить свой образ жизни: займитесь спортом, начните ходить на концерты или театральные постановки. 4. Найдите спутника по интересам, который сможет вас поддержать и мотивировать.


# Полный Fine Tuning

In [ ]:
import torch

from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

training_args = SFTConfig(
    output_dir="./qwen-full-finetuning",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    max_length=512,
)

trainer = SFTTrainer(
    model=MODEL_NAME,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

# Посмотрим, сколько параметров обучается
model = trainer.model

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Всего параметров:", total_params)
print("Обучаемых параметров:", trainable_params)
print("Доля обучаемых, %:", round(100 * trainable_params / total_params, 2))

trainer.train()

trainer.save_model("./qwen-full-finetuned")
tokenizer.save_pretrained("./qwen-full-finetuned")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/2983 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2983 [00:00<?, ? examples/s]

[transformers] The model is already on multiple devices. Skipping the move to device specified in `args`.


Всего параметров: 494032768
Обучаемых параметров: 494032768
Доля обучаемых, %: 100.0


ValueError: Pointer argument (at 0) cannot be accessed from Triton (cpu tensor?)

# LoRA

In [ ]:
import torch
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer

from transformers import AutoTokenizer


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

training_args = SFTConfig(
    output_dir="./qwen-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    fp16=False,
    max_length=512,
)

trainer = SFTTrainer(
    model=MODEL_NAME,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

model = trainer.model

total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Всего параметров:", total_params)
print("Обучаемых параметров:", trainable_params)
print("Доля обучаемых, %:", round(100 * trainable_params / total_params, 2))

trainer.train()

# Сохраняются адаптеры, а не полная копия базовой модели
trainer.save_model("./qwen-lora-adapter")
tokenizer.save_pretrained("./qwen-lora-adapter")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/2983 [00:00<?, ? examples/s]

Всего параметров: 494573440
Обучаемых параметров: 540672
Доля обучаемых, %: 0.11


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.347508,1.387561,1.553220,0.669755,3476996.000000


('./qwen-lora-adapter/tokenizer_config.json',
 './qwen-lora-adapter/chat_template.jinja',
 './qwen-lora-adapter/tokenizer.json')

# QLoRA

In [ ]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

if not torch.cuda.is_available():
    raise RuntimeError(
        "Для этого примера QLoRA требуется совместимая GPU-среда."
    )


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./qwen-qlora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    bf16=False,
    fp16=False,
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

trainer.save_model("./qwen-qlora-adapter")
tokenizer.save_pretrained("./qwen-qlora-adapter")

Adding EOS to train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/26839 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/2983 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/2983 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2983 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2983 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained("./qwen-lora-adapter",torch_dtype=torch.float16)
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./qwen-lora-merged", safe_serialization=True)
tokenizer.save_pretrained("./qwen-lora-merged")